## Imports

In [ ]:
import tensorflow.keras as keras
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from keras.models import Sequential
from keras.layers import Dense, Input, Flatten, Conv2D, MaxPooling2D
from keras.utils import plot_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## Overview of Keras
The following are the common steps you want to take when fitting a neural network using keras:
1. Load data
2. Define keras model
3. Compile model
4. Fit model
5. Evaluate model
6. Use model for prediction

## References

- [Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow, 3rd Edition]( https://learning.oreilly.com/library/view/hands-on-machine-learning/9781098125967/ )


## Example - Number Recognition

### Load data

We will be looking at an example dataset from keras called MNIST, which is a collection of images of the digits 0-9.

In [ ]:
mnist = tf.keras.datasets.mnist
(x_train, y_train),(x_test, y_test) = mnist.load_data() # Automatically shuffles data into random training and testing sets

In [ ]:
(
  type(x_train),
  type(y_train),
  type(x_test),
  type(y_test)
)


In [ ]:
(
  x_train.shape,
  y_train.shape,
  x_test.shape,
  y_test.shape
)


In [ ]:
x_train.shape # 60,000 images that are 28 x 28 pixels

In [ ]:
# shape of a single image
x_train.shape[1:]

The first image as a matrix of pixel intensities.

In [ ]:
x_train[0]

The 11th row of the first image.

In [ ]:
x_train[0][10]

The pixel intensity of the 10th column of the 11th row of the first image.


In [ ]:
x_train[0][10][9]

In [ ]:
# Visualize one of the training samples
plt.imshow( x_train[0], cmap = plt.cm.gray_r )
plt.show()

In [ ]:
# See the response for that sample
y_train[0]

### Fitting an ANN

#### Define keras model

First, we will normalize our data values to fall between 0 and 1.

In [ ]:
# View before normalization
x_train[0][10]

In [ ]:
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)

In [ ]:
# View after normalization
x_train[0][10]

In [ ]:
plt.imshow(x_train[0],cmap=plt.cm.gray_r)
plt.show()

Start the model as a feed forward (sequential) model.

In [ ]:
model = Sequential()

# Flatten input data into a 1D structure
model.add(Flatten())

# Define first hidden layers
model.add(
  Dense(
    name = "hidden1",
    units = 144,
    activation = 'relu' ,
  )
)

# Add second hidden layer
model.add(
  Dense(
    name = "hidden2",
    units = 144,
    activation = 'relu' ,
  )
)

# Define output layer
model.add(
  Dense(
    name = "output",
    units = 10,
    activation = 'softmax' ,
  )
)

#### Compile model

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy'],
)

#### Fit model

In [ ]:
model.fit(x_train, y_train, epochs=8) ;

In [ ]:
model.summary()

In [ ]:
plot_model(model)

#### Evaluate model

In [ ]:
val_loss, val_acc = model.evaluate(x_test, y_test)
val_acc

#### Use model for prediction

In [ ]:
predictions = model.predict(x_test)
print(predictions[257])

In [ ]:
[ print(f"{k}: {x:.2f}") for k, x in enumerate(predictions[257]*100) ]


In [ ]:
x_test.shape

In [ ]:
predictions.shape

In [ ]:
print(np.argmax(predictions[257]))

In [ ]:
y_test[257]

In [ ]:
plt.imshow(x_test[257], cmap=plt.cm.gray_r)
plt.show()

### Fitting a CNN

[Max pooling example](https://deeplizard.com/learn/video/ZjM_XQa5s6s)

In [ ]:
x_train.shape

In [ ]:
x_train[0][10][9]

In [ ]:
# Reshape data to be 4 dimensional
# long method: x_train2 = x_train.reshape( x_train.shape[0], x_train.shape[1], x_train.shape[2], 1)
x_train2 = x_train.reshape( x_train.shape + (1,) )
x_train2.shape

In [ ]:
# shape of a single image
x_train2.shape[1:]

In [ ]:
# notice vector instead of scalar
x_train2[0][10][9]

#### Define Keras Model


In [ ]:
model = Sequential()


In [ ]:
# Define input layer
model.add(
  Input(
    shape=x_train2.shape[1:],
    name="model_input",
  ),
)


In [ ]:
# Define first hidden layer
model.add(
  Conv2D(
    name = "Conv1",
    filters = 64,
    kernel_size = (3,3),
    activation = 'relu',
  )
)

model.add(
    MaxPooling2D(pool_size=(2,2))
)


In [ ]:
# Define second hidden layer
model.add(
  Conv2D(
    name = "Conv2",
    filters = 64,
    kernel_size = (3,3),
    activation = 'relu',
  )
)

model.add(
  MaxPooling2D(pool_size=(2,2))
)


In [ ]:
# Flatten data to be used in output layer
model.add(
    Flatten()
)

# Define output layer
model.add(
  Dense(
    name = "output",
    units = 10,
    activation = 'softmax',
  )
)


#### Compile model

In [ ]:
# Compile model
model.compile(
  optimizer = 'adam',
  loss = 'sparse_categorical_crossentropy',
  metrics = ['accuracy'],
)


#### Fit model

In [ ]:
# Fit model
model.fit(x_train2, y_train, epochs=8)

In [ ]:
model.fit(x_train2, y_train, epochs=2)


In [ ]:
model.summary()

In [ ]:
plot_model(model)

#### Evaluate model

In [ ]:
# Change test set to 4-D
x_test2 = x_test.reshape( x_test.shape + (1,) )

# Evaluate model
val_loss, val_acc = model.evaluate(x_test2, y_test)
print(val_loss)
print(val_acc)

#### Use model for prediction

In [ ]:
# make predictions based on testing set
predictions = model.predict(x_test2)


In [ ]:
predictions.shape

In [ ]:
i = 10
filter = ( (predictions).argmax( axis = 1 ) != y_test )
plt.imshow(x_test[ filter ][i], cmap=plt.cm.gray_r)
list(enumerate(predictions[filter][i]*100)), y_test[filter][i]


In [ ]:
# pick any image from 0 to 10k-1
i = 257
# i = 160
# i = 2982


for k, x in enumerate(predictions[i]*100):
  print(f"{k}: {x:.2f}")
print(np.argmax(predictions[i]))
print(y_test[i])
plt.imshow(x_test[i], cmap=plt.cm.gray_r)
plt.show()

In [ ]:
seed=26   # other numbers to try: 27, 29, 30

rng = np.random.default_rng(seed=seed)
numbers = rng.choice(len(x_test), size=20, replace=False)

for i in np.sort(numbers):
  a = y_test[i]
  p = np.argmax(predictions[i])
  print(f"    index: {i}\n   actual: {a}\npredicted: {p}{ '  << == miss' if a != p else ''}")
  plt.imshow(x_test[i], cmap=plt.cm.gray_r)
  plt.show()
  print("\n")
